In [ ]:
from pathlib import Path
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ROOT = Path(r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump")

RAW_SOIL = ROOT / "data" / "raw" / "soil"
RAW_CLIMATE = ROOT / "data" / "raw" / "climate"
PROCESSED_CLIMATE = ROOT / "data" / "processed" / "climate"

BOUNDARY_FILE = RAW_SOIL / "district_boundary" / "IND_ADM2.geojson"
CROP_FILE = ROOT / "data" / "processed" / "unified" / "unified_crop_yield_2013_2025.csv"

RAW_CLIMATE.mkdir(parents=True, exist_ok=True)
PROCESSED_CLIMATE.mkdir(parents=True, exist_ok=True)

CACHE_DIR = RAW_CLIMATE / "nasa_power_monthly_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2013
END_YEAR = 2025

print("Project root:", ROOT)
print("Boundary file exists:", BOUNDARY_FILE.exists())
print("Crop dataset exists:", CROP_FILE.exists())
print("Climate cache:", CACHE_DIR)


Project root: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump
Boundary file exists: True
Crop dataset exists: True
Climate cache: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\climate\nasa_power_monthly_cache


In [ ]:
#  Loading and validating the district boundary dataset
if not BOUNDARY_FILE.exists():
    raise FileNotFoundError(
        f"District boundary file not found:\n{BOUNDARY_FILE}"
    )

districts = gpd.read_file(BOUNDARY_FILE)

required_boundary_cols = {"shapeName", "shapeID", "geometry"}
missing = required_boundary_cols - set(districts.columns)

if missing:
    raise ValueError(
        f"Boundary dataset is missing required columns: {sorted(missing)}"
    )

if districts.crs is None:
    raise ValueError("Boundary dataset has no CRS.")

districts = districts.to_crs("EPSG:4326").copy()
districts["district"] = (
    districts["shapeName"]
    .astype(str)
    .str.strip()
)

districts = districts[
    districts.geometry.notna() & ~districts.geometry.is_empty
].copy()

# Fix invalid geometries if necessary.
invalid_count = (~districts.geometry.is_valid).sum()
if invalid_count:
    districts["geometry"] = districts.geometry.make_valid()

print("District boundary dataset loaded.")
print("Shape:", districts.shape)
print("CRS:", districts.crs)
print("Invalid geometries fixed:", invalid_count)
print("Unique district names:", districts["district"].nunique())
print("Duplicate district names:", districts["district"].duplicated().sum())

display(districts[["shapeName", "shapeID", "district", "geometry"]].head())


District boundary dataset loaded.
Shape: (735, 7)
CRS: EPSG:4326
Invalid geometries fixed: 0
Unique district names: 728
Duplicate district names: 7


,shapeName,shapeID,district,geometry
0,Ashoknagar,76128533B75548370501185,Ashoknagar,"POLYGON ((78.17491 24.84254, 78.17466 24.84194..."
1,Raisen,76128533B57893545331548,Raisen,"POLYGON ((77.38167 23.07004, 77.38094 23.0698,..."
2,Chhindwara,76128533B70646408240587,Chhindwara,"POLYGON ((79.23988 22.79135, 79.24 22.79172, 7..."
3,Betul,76128533B82559220423608,Betul,"POLYGON ((78.27229 22.39973, 78.27169 22.39897..."
4,Hoshangabad,76128533B45314020251888,Hoshangabad,"POLYGON ((78.03027 22.79953, 78.02986 22.79969..."


In [ ]:
# Load crop master keys for cross-checking

if not CROP_FILE.exists():
    raise FileNotFoundError(
        f"Unified crop dataset not found:\n{CROP_FILE}"
    )

crop_keys = pd.read_csv(
    CROP_FILE,
    usecols=["state", "district"]
).drop_duplicates()

crop_keys["state"] = crop_keys["state"].astype(str).str.strip()
crop_keys["district"] = crop_keys["district"].astype(str).str.strip()

# A district name may be unique to one state, or ambiguous across states.
district_state_map = (
    crop_keys.groupby("district")["state"]
    .agg(lambda s: sorted(set(s.dropna())))
    .to_dict()
)

def unique_state(district_name):
    states = district_state_map.get(district_name, [])
    return states[0] if len(states) == 1 else np.nan

districts["state"] = districts["district"].map(unique_state)

print("Crop master loaded.")
print("Unique crop state-district combinations:", len(crop_keys))
print("Boundary districts with uniquely inferred state:",
      districts["state"].notna().sum())
print("Boundary districts without a unique state mapping:",
      districts["state"].isna().sum())

ambiguous_crop_districts = [
    d for d, states in district_state_map.items() if len(states) > 1
]

print("Crop district names occurring in multiple states:",
      len(ambiguous_crop_districts))

if ambiguous_crop_districts:
    display(
        crop_keys[
            crop_keys["district"].isin(ambiguous_crop_districts)
        ]
        .sort_values(["district", "state"])
        .head(30)
    )


Crop master loaded.
Unique crop state-district combinations: 806
Boundary districts with uniquely inferred state: 618
Boundary districts without a unique state mapping: 117
Crop district names occurring in multiple states: 3


,state,district
742,Chhattisgarh,Bilaspur
1305,Himachal Pradesh,Bilaspur
1317,Himachal Pradesh,Hamirpur
4363,Uttar Pradesh,Hamirpur
3531,Rajasthan,Pratapgarh
4613,Uttar Pradesh,Pratapgarh


In [ ]:
# Creating stable representative points for API queries

district_points = districts.copy()
district_points["query_point"] = district_points.geometry.representative_point()

district_points["latitude"] = district_points["query_point"].y.astype(float)
district_points["longitude"] = district_points["query_point"].x.astype(float)

# Keeping the original boundary geometry out of the climate table.
point_metadata = district_points[
    ["shapeID", "district", "state", "latitude", "longitude"]
].copy()

# Ensuring coordinates are valid geographic coordinates.
if (
    ~point_metadata["latitude"].between(-90, 90).all()
    or ~point_metadata["longitude"].between(-180, 180).all()
):
    raise ValueError("One or more generated coordinates are outside valid ranges.")

print("Representative query points created.")
print("Districts:", len(point_metadata))
display(point_metadata.head(10))


Representative query points created.
Districts: 735


,shapeID,district,state,latitude,longitude
0,76128533B75548370501185,Ashoknagar,Madhya Pradesh,24.614385,77.875152
1,76128533B57893545331548,Raisen,Madhya Pradesh,23.265145,78.174833
2,76128533B70646408240587,Chhindwara,Madhya Pradesh,22.138885,78.810870
3,76128533B82559220423608,Betul,Madhya Pradesh,21.878745,77.878124
4,76128533B45314020251888,Hoshangabad,NaN,22.597865,77.928911
5,76128533B39959897010345,Sehore,Madhya Pradesh,23.112075,77.069023
6,76128533B27615708126939,Jabalpur,Madhya Pradesh,23.223455,79.986373
7,76128533B20683890392439,Narsimhapur,Madhya Pradesh,22.931940,79.089024
8,76128533B81607600692021,Panna,Madhya Pradesh,24.447310,80.148199
9,76128533B89286042589317,Ujjain,Madhya Pradesh,23.291325,75.626912


In [ ]:
# API configuration and test request

API_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"

PARAMETERS = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "WS10M",
    "PRECTOTCORR",
    "PRECTOTCORR_SUM",
    "ALLSKY_SFC_SW_DWN",
]

params = {
    "parameters": ",".join(PARAMETERS),
    "community": "AG",
    "longitude": float(point_metadata.iloc[0]["longitude"]),
    "latitude": float(point_metadata.iloc[0]["latitude"]),
    "start": START_YEAR,
    "end": END_YEAR,
    "format": "JSON",
}

print("Testing NASA POWER API...")
print("Test district:", point_metadata.iloc[0]["district"])
print("Coordinates:",
      round(params["latitude"], 5),
      round(params["longitude"], 5))

response = requests.get(API_URL, params=params, timeout=60)
response.raise_for_status()

test_json = response.json()

if "properties" not in test_json or "parameter" not in test_json["properties"]:
    raise ValueError("NASA POWER response does not contain the expected parameter structure.")

returned_parameters = set(test_json["properties"]["parameter"].keys())
missing_parameters = set(PARAMETERS) - returned_parameters

print("\nAPI test successful.")
print("Returned parameters:", sorted(returned_parameters))

if missing_parameters:
    print("Warning — parameters not returned:", sorted(missing_parameters))
else:
    print("All requested parameters were returned.")


Testing NASA POWER API...
Test district: Ashoknagar
Coordinates: 24.61438 77.87515

API test successful.
Returned parameters: ['ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'PRECTOTCORR_SUM', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS10M']
All requested parameters were returned.


In [ ]:
#Request helper with retry logic

def make_session():
    session = requests.Session()

    retry = Retry(
        total=4,
        connect=4,
        read=4,
        status=4,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=8,
        pool_maxsize=8,
    )

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


def cache_file(shape_id):
    safe_id = str(shape_id).replace("/", "_").replace("\\", "_")
    return CACHE_DIR / f"{safe_id}.json"


def fetch_district_climate(row):
    shape_id = row["shapeID"]
    cache_path = cache_file(shape_id)

    # Reuse an already successful download.
    if cache_path.exists():
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                cached = json.load(f)

            if (
                "properties" in cached
                and "parameter" in cached["properties"]
            ):
                return {
                    "shapeID": shape_id,
                    "district": row["district"],
                    "state": row["state"],
                    "latitude": row["latitude"],
                    "longitude": row["longitude"],
                    "json": cached,
                    "status": "cached",
                    "error": None,
                }
        except Exception:
            pass

    request_params = {
        "parameters": ",".join(PARAMETERS),
        "community": "AG",
        "longitude": float(row["longitude"]),
        "latitude": float(row["latitude"]),
        "start": START_YEAR,
        "end": END_YEAR,
        "format": "JSON",
    }

    session = make_session()

    try:
        response = session.get(
            API_URL,
            params=request_params,
            timeout=90,
        )

        response.raise_for_status()
        data = response.json()

        if (
            "properties" not in data
            or "parameter" not in data["properties"]
        ):
            raise ValueError("Unexpected NASA POWER response structure.")

        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(data, f)

        return {
            "shapeID": shape_id,
            "district": row["district"],
            "state": row["state"],
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "json": data,
            "status": "downloaded",
            "error": None,
        }

    except Exception as exc:
        return {
            "shapeID": shape_id,
            "district": row["district"],
            "state": row["state"],
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "json": None,
            "status": "failed",
            "error": repr(exc),
        }


In [7]:
# Downloading climate data for all districts

# One monthly API request returns the complete 2013–2025 monthly time series for one district point. Therefore the full run is approximately one request per boundary district, not one request per month.

MAX_WORKERS = 4

records = []
failures = []

rows = point_metadata.to_dict("records")
total = len(rows)

print(f"Starting climate download for {total} district points...")
print(f"Period: {START_YEAR}–{END_YEAR}")
print(f"Parallel workers: {MAX_WORKERS}")
print(f"Existing cache files: {len(list(CACHE_DIR.glob('*.json')))}")
print()

start_time = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(fetch_district_climate, row): row
        for row in rows
    }

    completed = 0

    for future in as_completed(futures):
        result = future.result()
        completed += 1

        if result["status"] == "failed":
            failures.append(result)
        else:
            records.append(result)

        if completed % 25 == 0 or completed == total:
            elapsed = time.time() - start_time
            print(
                f"Progress: {completed}/{total} | "
                f"success={len(records)} | "
                f"failed={len(failures)} | "
                f"elapsed={elapsed/60:.1f} min"
            )

print("\nDownload phase completed.")
print("Successful:", len(records))
print("Failed:", len(failures))

if failures:
    failure_df = pd.DataFrame(failures)
    display(failure_df[["shapeID", "district", "error"]].head(20))
else:
    failure_df = pd.DataFrame()

if len(records) == 0:
    raise RuntimeError(
        "No climate records were downloaded successfully. "
        "Check the API test above and your internet connection."
    )


Starting climate download for 735 district points...
Period: 2013–2025
Parallel workers: 4
Existing cache files: 0

Progress: 25/735 | success=25 | failed=0 | elapsed=0.2 min
Progress: 50/735 | success=50 | failed=0 | elapsed=0.3 min
Progress: 75/735 | success=75 | failed=0 | elapsed=0.5 min
Progress: 100/735 | success=100 | failed=0 | elapsed=0.7 min
Progress: 125/735 | success=125 | failed=0 | elapsed=0.9 min
Progress: 150/735 | success=150 | failed=0 | elapsed=1.0 min
Progress: 175/735 | success=175 | failed=0 | elapsed=1.2 min
Progress: 200/735 | success=200 | failed=0 | elapsed=1.4 min
Progress: 225/735 | success=225 | failed=0 | elapsed=1.6 min
Progress: 250/735 | success=250 | failed=0 | elapsed=1.8 min
Progress: 275/735 | success=275 | failed=0 | elapsed=2.0 min
Progress: 300/735 | success=300 | failed=0 | elapsed=2.2 min
Progress: 325/735 | success=325 | failed=0 | elapsed=2.3 min
Progress: 350/735 | success=350 | failed=0 | elapsed=2.5 min
Progress: 375/735 | success=375 | fa

,shapeID,district,error
0,76128533B23858286755005,Alipurduar,HTTPError('429 Client Error: Too Many Requests...
1,76128533B24861976740713,Jhargram,HTTPError('429 Client Error: Too Many Requests...
2,76128533B60845263086042,Paschim Barddhaman,HTTPError('429 Client Error: Too Many Requests...
3,76128533B78458325839103,Kalimpong,HTTPError('429 Client Error: Too Many Requests...
4,76128533B14968875277413,Narayanpet,HTTPError('429 Client Error: Too Many Requests...
5,76128533B84398038505400,Mulugu,HTTPError('429 Client Error: Too Many Requests...
6,76128533B38767180038480,Niwari,HTTPError('429 Client Error: Too Many Requests...
7,76128533B24017517661394,Pakke Kessang,HTTPError('429 Client Error: Too Many Requests...
8,76128533B19898496046456,Kamle,HTTPError('429 Client Error: Too Many Requests...
9,76128533B46084756537322,Shi Yomi,HTTPError('429 Client Error: Too Many Requests...


In [8]:
#  Saving failure log and parsing monthly records

failure_log = PROCESSED_CLIMATE / "nasa_power_failed_requests.csv"

if not failure_df.empty:
    failure_df[
        ["shapeID", "district", "state", "latitude", "longitude", "error"]
    ].to_csv(failure_log, index=False)
    print("Failure log saved:", failure_log)
else:
    if failure_log.exists():
        failure_log.unlink()
    print("No failed requests.")

def parse_power_response(result):
    data = result["json"]
    parameter_data = data["properties"]["parameter"]

    time_keys = sorted(
        k for k in parameter_data.get("T2M", {}).keys()
        if len(k) == 6 and k.isdigit() and 1 <= int(k[-2:]) <= 12
    )

    output = []

    for key in time_keys:
        year = int(key[:4])
        month = int(key[4:])

        row = {
            "shapeID": result["shapeID"],
            "district": result["district"],
            "state": result["state"],
            "latitude": result["latitude"],
            "longitude": result["longitude"],
            "year": year,
            "month": month,
        }

        for parameter in PARAMETERS:
            values = parameter_data.get(parameter, {})
            row[parameter] = values.get(key, np.nan)

        output.append(row)

    return output


monthly_records = []

for result in records:
    monthly_records.extend(parse_power_response(result))

climate_monthly = pd.DataFrame(monthly_records)

if climate_monthly.empty:
    raise RuntimeError("No monthly climate rows were parsed.")

climate_monthly = climate_monthly.sort_values(
    ["shapeID", "year", "month"]
).reset_index(drop=True)

print("Monthly climate dataset parsed.")
print("Shape:", climate_monthly.shape)
print("Years:", climate_monthly["year"].min(), "to", climate_monthly["year"].max())
print("Districts:", climate_monthly["shapeID"].nunique())
print("Expected months per complete district-year:", 12)

display(climate_monthly.head(12))

Failure log saved: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\climate\nasa_power_failed_requests.csv
Monthly climate dataset parsed.
Shape: (111540, 15)
Years: 2013 to 2025
Districts: 715
Expected months per complete district-year: 12


,shapeID,district,state,latitude,longitude,year,month,T2M,T2M_MAX,T2M_MIN,RH2M,WS10M,PRECTOTCORR,PRECTOTCORR_SUM,ALLSKY_SFC_SW_DWN
0,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,1,-3.04,8.05,-13.33,41.93,2.42,0.11,3.45,14.53
1,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,2,-1.03,9.96,-16.72,49.58,2.36,1.10,30.80,14.55
2,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,3,2.45,11.87,-6.08,57.50,2.62,0.55,17.00,16.25
3,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,4,5.14,15.71,-3.31,63.93,2.80,1.40,42.07,16.43
4,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,5,8.40,18.89,0.17,69.57,2.66,4.67,144.68,16.01
5,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,6,11.31,18.88,5.27,86.54,2.41,6.52,195.49,15.89
6,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,7,11.88,16.72,7.50,90.68,2.57,8.32,258.04,14.73
7,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,8,11.22,16.37,7.28,90.83,2.32,9.33,289.26,15.21
8,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,9,9.74,15.66,3.03,87.96,2.05,4.13,123.99,15.28
9,76128533B10270619600482,South District,NaN,27.3072,88.383434,2013,10,6.38,14.16,-0.92,84.98,1.87,4.24,131.35,13.00


In [9]:
#  Clean missing-value codes and numeric fields

# NASA POWER commonly uses -999 as its missing/fill value.
numeric_cols = PARAMETERS

for col in numeric_cols:
    climate_monthly[col] = pd.to_numeric(
        climate_monthly[col],
        errors="coerce"
    )
    climate_monthly.loc[
        climate_monthly[col] <= -900,
        col
    ] = np.nan

# Convert coordinates and identifiers to consistent types.
climate_monthly["latitude"] = pd.to_numeric(
    climate_monthly["latitude"], errors="coerce"
)
climate_monthly["longitude"] = pd.to_numeric(
    climate_monthly["longitude"], errors="coerce"
)

# Check that all requested years/months are present.
climate_monthly["date"] = pd.to_datetime(
    climate_monthly["year"].astype(str)
    + "-"
    + climate_monthly["month"].astype(str).str.zfill(2)
    + "-01"
)

print("Cleaning completed.")
print("\nMissing values by parameter:")
display(climate_monthly[numeric_cols].isna().sum())

print("\nDate range:")
print(climate_monthly["date"].min(), "to", climate_monthly["date"].max())


Cleaning completed.

Missing values by parameter:


T2M                  0
T2M_MAX              0
T2M_MIN              0
RH2M                 0
WS10M                0
PRECTOTCORR          0
PRECTOTCORR_SUM      0
ALLSKY_SFC_SW_DWN    0
dtype: int64


Date range:
2013-01-01 00:00:00 to 2025-12-01 00:00:00


In [10]:
# Convert precipitation rate to a monthly rainfall check

# PRECTOTCORR is a precipitation rate in mm/day.
# PRECTOTCORR_SUM is the monthly precipitation sum in mm.

# We keep both variables. For agricultural annual/seasonal rainfall totals, PRECTOTCORR_SUM is preferred.

climate_monthly["rainfall_mm"] = climate_monthly["PRECTOTCORR_SUM"]

# If a monthly sum is missing but the rate exists, calculate an estimate using the actual number of days in that month.
days_in_month = climate_monthly["date"].dt.days_in_month

fallback = (
    climate_monthly["rainfall_mm"].isna()
    & climate_monthly["PRECTOTCORR"].notna()
)

climate_monthly.loc[fallback, "rainfall_mm"] = (
    climate_monthly.loc[fallback, "PRECTOTCORR"]
    * days_in_month[fallback]
)

print("Rainfall feature created.")
print("Rows using precipitation-rate fallback:", int(fallback.sum()))

display(
    climate_monthly[
        [
            "district", "year", "month",
            "PRECTOTCORR",
            "PRECTOTCORR_SUM",
            "rainfall_mm"
        ]
    ].head(12)
)


Rainfall feature created.
Rows using precipitation-rate fallback: 0


,district,year,month,PRECTOTCORR,PRECTOTCORR_SUM,rainfall_mm
0,South District,2013,1,0.11,3.45,3.45
1,South District,2013,2,1.10,30.80,30.80
2,South District,2013,3,0.55,17.00,17.00
3,South District,2013,4,1.40,42.07,42.07
4,South District,2013,5,4.67,144.68,144.68
5,South District,2013,6,6.52,195.49,195.49
6,South District,2013,7,8.32,258.04,258.04
7,South District,2013,8,9.33,289.26,289.26
8,South District,2013,9,4.13,123.99,123.99
9,South District,2013,10,4.24,131.35,131.35


In [11]:
# Create agricultural climate features

# Monsoon definition used for this project: June–September (JJAS), the standard southwest-monsoon season.

climate_monthly["is_monsoon"] = climate_monthly["month"].isin(
    [6, 7, 8, 9]
)

annual = (
    climate_monthly
    .groupby(
        ["shapeID", "district", "state", "latitude", "longitude", "year"],
        as_index=False
    )
    .agg(
        annual_rainfall_mm=("rainfall_mm", "sum"),
        annual_mean_temp_c=("T2M", "mean"),
        annual_max_temp_c=("T2M_MAX", "mean"),
        annual_min_temp_c=("T2M_MIN", "mean"),
        annual_relative_humidity_pct=("RH2M", "mean"),
        annual_wind_speed_m_s=("WS10M", "mean"),
        annual_solar_radiation=("ALLSKY_SFC_SW_DWN", "mean"),
    )
)

monsoon = (
    climate_monthly[climate_monthly["is_monsoon"]]
    .groupby(
        ["shapeID", "year"],
        as_index=False
    )
    .agg(
        monsoon_rainfall_mm=("rainfall_mm", "sum"),
        monsoon_mean_temp_c=("T2M", "mean"),
        monsoon_max_temp_c=("T2M_MAX", "mean"),
        monsoon_min_temp_c=("T2M_MIN", "mean"),
        monsoon_relative_humidity_pct=("RH2M", "mean"),
        monsoon_wind_speed_m_s=("WS10M", "mean"),
        monsoon_solar_radiation=("ALLSKY_SFC_SW_DWN", "mean"),
    )
)

climate_annual = annual.merge(
    monsoon,
    on=["shapeID", "year"],
    how="left"
)

climate_annual = climate_annual.sort_values(
    ["shapeID", "year"]
).reset_index(drop=True)

print("Annual agricultural climate features created.")
print("Shape:", climate_annual.shape)
display(climate_annual.head(12))


Annual agricultural climate features created.
Shape: (7800, 20)


,shapeID,district,state,latitude,longitude,year,annual_rainfall_mm,annual_mean_temp_c,annual_max_temp_c,annual_min_temp_c,annual_relative_humidity_pct,annual_wind_speed_m_s,annual_solar_radiation,monsoon_rainfall_mm,monsoon_mean_temp_c,monsoon_max_temp_c,monsoon_min_temp_c,monsoon_relative_humidity_pct,monsoon_wind_speed_m_s,monsoon_solar_radiation
0,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2013,565.09,25.444167,36.595000,14.418333,43.744167,2.825000,17.917500,456.40,32.2550,41.8625,24.1425,56.3275,2.8625,20.4650
1,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2014,349.27,25.660000,37.511667,15.180000,38.374167,2.828333,17.877500,237.64,33.9150,43.2400,25.2800,43.6875,3.4150,21.2450
2,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2015,599.18,25.321667,37.081667,14.384167,42.894167,2.760833,17.615000,405.54,31.7950,41.5975,23.1675,52.5325,3.0475,20.2975
3,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2016,463.92,26.581667,38.025000,16.410000,36.270833,2.628333,17.718333,381.33,32.4825,41.5800,25.1425,54.2075,2.6250,20.2575
4,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2017,507.05,25.945833,37.981667,14.765000,39.492500,2.692500,17.757500,383.99,32.1575,41.8975,24.1075,53.6175,2.9300,20.0825
5,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2018,526.78,25.769167,37.681667,15.335000,41.024167,2.630000,17.404167,473.88,32.0975,41.7500,24.1525,56.8425,2.8075,18.7475
6,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2019,535.92,25.186667,37.362500,14.579167,47.049167,2.701667,16.957500,368.85,33.0025,43.0950,24.5100,54.2325,2.7850,19.8925
7,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2020,597.18,24.959167,37.155833,14.156667,48.704167,2.627500,17.585833,384.59,32.7375,43.0225,24.0350,56.3900,2.6575,20.3075
8,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2021,668.65,25.190000,36.854167,14.913333,47.527500,2.653333,17.070833,554.00,31.6600,40.5900,24.3950,61.3150,2.7025,18.9075
9,76128533B10727492375508,Fatehabad,Haryana,29.535655,75.575515,2022,727.83,25.575833,37.097500,15.129167,49.880833,2.761667,17.250000,575.85,31.4825,40.0400,23.6350,61.9975,2.9575,19.1350


In [12]:
#Validation checks

print("CLIMATE DATA VALIDATION")

print("\nMonthly shape:")
print(climate_monthly.shape)

print("\nAnnual shape:")
print(climate_annual.shape)

print("\nUnique districts:")
print("Monthly:", climate_monthly["shapeID"].nunique())
print("Annual:", climate_annual["shapeID"].nunique())

print("\nYear coverage:")
print(
    climate_monthly["year"].min(),
    "to",
    climate_monthly["year"].max()
)

print("\nDuplicate monthly keys:")
monthly_dupes = climate_monthly.duplicated(
    subset=["shapeID", "year", "month"]
).sum()
print(monthly_dupes)

print("\nDuplicate annual keys:")
annual_dupes = climate_annual.duplicated(
    subset=["shapeID", "year"]
).sum()
print(annual_dupes)

print("\nMonthly rows per district statistics:")
rows_per_district = climate_monthly.groupby("shapeID").size()
display(rows_per_district.describe())

print("\nAnnual rows per district statistics:")
annual_rows_per_district = climate_annual.groupby("shapeID").size()
display(annual_rows_per_district.describe())

print("\nMissing climate values:")
display(
    climate_monthly[
        PARAMETERS + ["rainfall_mm"]
    ].isna().sum()
)

print("\nBasic plausibility ranges:")
display(
    climate_monthly[
        ["T2M", "T2M_MIN", "T2M_MAX", "RH2M",
         "WS10M", "rainfall_mm", "ALLSKY_SFC_SW_DWN"]
    ].describe()
)

if monthly_dupes != 0 or annual_dupes != 0:
    raise ValueError("Duplicate climate keys detected.")

if not climate_monthly["month"].between(1, 12).all():
    raise ValueError("Invalid month values detected.")



CLIMATE DATA VALIDATION

Monthly shape:
(111540, 18)

Annual shape:
(7800, 20)

Unique districts:
Monthly: 715
Annual: 600

Year coverage:
2013 to 2025

Duplicate monthly keys:
0

Duplicate annual keys:
0

Monthly rows per district statistics:


count    715.0
mean     156.0
std        0.0
min      156.0
25%      156.0
50%      156.0
75%      156.0
max      156.0
dtype: float64


Annual rows per district statistics:


count    600.0
mean      13.0
std        0.0
min       13.0
25%       13.0
50%       13.0
75%       13.0
max       13.0
dtype: float64


Missing climate values:


T2M                  0
T2M_MAX              0
T2M_MIN              0
RH2M                 0
WS10M                0
PRECTOTCORR          0
PRECTOTCORR_SUM      0
ALLSKY_SFC_SW_DWN    0
rainfall_mm          0
dtype: int64


Basic plausibility ranges:


,T2M,T2M_MIN,T2M_MAX,RH2M,WS10M,rainfall_mm,ALLSKY_SFC_SW_DWN
count,111540.000000,111540.000000,111540.000000,111540.000000,111540.000000,111540.000000,111540.000000
mean,24.149271,15.743873,33.576772,62.816546,2.889210,110.389284,17.388091
std,7.138726,8.334041,7.340740,20.451327,1.085477,149.485260,3.873116
min,-22.230000,-40.050000,-10.550000,12.020000,0.600000,0.000000,4.520000
25%,20.380000,10.030000,29.720000,46.400000,2.220000,6.260000,14.740000
50%,25.450000,17.440000,33.210000,66.970000,2.780000,44.140000,16.910000
75%,28.680000,22.460000,38.290000,80.990000,3.390000,169.522500,20.070000
max,39.280000,30.900000,49.840000,95.400000,9.060000,2173.840000,30.250000


In [13]:
#Cross-check climate coverage against crop master

crop_coverage = (
    crop_keys
    .groupby(["state", "district"])
    .size()
    .reset_index(name="crop_records")
)

climate_coverage = (
    climate_monthly[
        ["shapeID", "state", "district", "year"]
    ]
    .drop_duplicates()
)

print("Crop state-district combinations:", len(crop_coverage))
print("Climate district-year combinations:", len(climate_coverage))

# District names that have an unambiguous state mapping can be compared.
climate_unique_state = climate_monthly[
    climate_monthly["state"].notna()
][["state", "district"]].drop_duplicates()

crop_state_keys = set(
    zip(crop_coverage["state"], crop_coverage["district"])
)
climate_state_keys = set(
    zip(
        climate_unique_state["state"],
        climate_unique_state["district"]
    )
)

matched_keys = crop_state_keys & climate_state_keys

print("Crop keys with a unique climate-state match:", len(matched_keys))
print(
    "Crop keys without an unambiguous climate-state match:",
    len(crop_state_keys - climate_state_keys)
)

print(
    "\nNote: state-district merging is intentionally NOT performed here. "
    "The boundary dataset has duplicate district names and no state field. "
    "The next integration notebook will handle the final join carefully."
)


Crop state-district combinations: 806
Climate district-year combinations: 9295
Crop keys with a unique climate-state match: 596
Crop keys without an unambiguous climate-state match: 210

Note: state-district merging is intentionally NOT performed here. The boundary dataset has duplicate district names and no state field. The next integration notebook will handle the final join carefully.


In [14]:
#Save processed climate datasets

monthly_output = (
    PROCESSED_CLIMATE
    / "district_climate_monthly_nasa_power_2013_2025.csv"
)

annual_output = (
    PROCESSED_CLIMATE
    / "district_climate_annual_nasa_power_2013_2025.csv"
)

# Drop helper columns not needed in the final monthly climate table.
monthly_final = climate_monthly.drop(
    columns=["date", "is_monsoon"],
    errors="ignore"
).copy()

monthly_final.to_csv(
    monthly_output,
    index=False
)

climate_annual.to_csv(
    annual_output,
    index=False
)

print("===== CLIMATE DATASETS SAVED =====")
print("\nMonthly file:")
print(monthly_output)
print("Shape:", monthly_final.shape)

print("\nAnnual file:")
print(annual_output)
print("Shape:", climate_annual.shape)

print("\nCache directory:")
print(CACHE_DIR)
print("Cached JSON files:", len(list(CACHE_DIR.glob("*.json"))))

print("\nFinal monthly columns:")
print(monthly_final.columns.tolist())

print("\nFinal annual columns:")
print(climate_annual.columns.tolist())


===== CLIMATE DATASETS SAVED =====

Monthly file:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\climate\district_climate_monthly_nasa_power_2013_2025.csv
Shape: (111540, 16)

Annual file:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\climate\district_climate_annual_nasa_power_2013_2025.csv
Shape: (7800, 20)

Cache directory:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\climate\nasa_power_monthly_cache
Cached JSON files: 715

Final monthly columns:
['shapeID', 'district', 'state', 'latitude', 'longitude', 'year', 'month', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'WS10M', 'PRECTOTCORR', 'PRECTOTCORR_SUM', 'ALLSKY_SFC_SW_DWN', 'rainfall_mm']

Final annual columns:
['shapeID', 'district', 'state', 'latitude', 'longitude', 'year', 'annual_rainfall_mm', 'annual_mean_temp_c', 'annual_max_temp_c', 'annual_min_temp_c', 'annual_relative_humidity_pct', 'annual_wind_speed_m_s', 'annual_so